# Módulo 0 — Datos climáticos históricos: Jocolí / Lavalle, Mendoza

**Fuente:** Open-Meteo Archive API  
**Período:** 1990–2024 (35 años)  
**Variables:** temperatura 2m · precipitación · evapotranspiración de referencia (ET₀)

In [1]:
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO_URL = "https://github.com/Emilialandgrebe/tesis-maestria.git"
    ROOT = Path("/content/tesis-maestria")
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt"), "-q"],
        check=True,
    )
else:
    ROOT = Path().resolve().parent  # notebooks/ -> raiz del proyecto

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Entorno : {'Google Colab' if EN_COLAB else 'local'}")
print(f"ROOT    : {ROOT}")

Entorno : Google Colab
ROOT    : /content/tesis-maestria


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import linregress
from statsmodels.nonparametric.smoothers_lowess import lowess

from src.data.climate_fetcher import fetch_climate_data
from src.data.climate_features import (
    calcular_calor_verano,
    calcular_deficit_hidrico,
    calcular_heladas_tardias,
    calcular_horas_frio,
)

np.random.seed(42)

df = fetch_climate_data()

horas_frio      = calcular_horas_frio(df)
heladas         = calcular_heladas_tardias(df)
calor_verano    = calcular_calor_verano(df)
deficit_hidrico = calcular_deficit_hidrico(df)

In [ ]:
print("=" * 60)
print("RANGO DE FECHAS")
print("=" * 60)
print(f"Inicio : {df.index.min()}")
print(f"Fin    : {df.index.max()}")
print(f"Total  : {len(df):,} registros horarios")
print(f"         ({len(df) / 8760:.1f} años aprox.)")

In [ ]:
print("PRIMERAS FILAS")
df.head()

In [ ]:
print("ÚLTIMAS FILAS")
df.tail()

In [ ]:
print("ESTRUCTURA DEL DATAFRAME")
df.info()

In [ ]:
print("ESTADÍSTICAS DESCRIPTIVAS")
df.describe()

In [ ]:
print("=" * 60)
print("VALORES NULOS POR COLUMNA")
print("=" * 60)
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(3)
resumen_nulos = nulos.to_frame("cantidad").assign(porcentaje=pct)
print(resumen_nulos)
print()
if nulos.sum() == 0:
    print("Sin valores nulos. Dataset completo.")
else:
    print(f"ATENCIÓN: {nulos.sum()} valores nulos en total.")

## Temperatura máxima media en verano (ene–feb)

In [ ]:
años = calor_verano.index.values.astype(float)
temperaturas = calor_verano.values

reg = linregress(años, temperaturas)
pendiente  = reg.slope
intercepto = reg.intercept
r2         = reg.rvalue ** 2
pvalor     = reg.pvalue

linea_tendencia = intercepto + pendiente * años

suavizado = lowess(temperaturas, años, frac=0.25)

desvio    = calor_verano.std()
banda_sup = temperaturas + desvio
banda_inf = temperaturas - desvio

umbral_extremo = 37
años_extremos  = calor_verano[calor_verano >= umbral_extremo]

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(años, temperaturas, color="#E67E22", linewidth=2,
        marker="o", markersize=4, label="Tmax media ene–feb")

ax.fill_between(años, banda_inf, banda_sup, color="#E67E22",
                alpha=0.15, label="±1 desvío estándar")

ax.fill_between(años, 35, 38, color="green",
                alpha=0.08, label="Rango óptimo pistacho (35–38 °C)")
ax.axhline(35, color="green", linewidth=0.7, linestyle="--")
ax.axhline(38, color="green", linewidth=0.7, linestyle="--")

ax.plot(años, linea_tendencia, color="#C0392B", linestyle="--", linewidth=2,
        label=f"Tendencia = {pendiente:.3f} °C/año\nR²={r2:.2f}   p={pvalor:.3f}")

ax.plot(suavizado[:, 0], suavizado[:, 1], color="black",
        linewidth=2, label="Suavizado LOWESS")

ax.axvspan(2015, 2024, color="gray", alpha=0.08, label="Última década")

ax.scatter(años_extremos.index, años_extremos.values, color="red",
           s=60, zorder=10, label=f"Años ≥ {umbral_extremo} °C")

ax.set_title(
    "Temperatura máxima diaria media del verano (enero–febrero)\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=14, fontweight="bold",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("Temperatura (°C)", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4)
ax.legend(loc="upper left")
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia.", ha="right",
            fontsize=9, color="gray")
plt.show()


## Días con temperatura extrema (ene–feb)

Frecuencia anual de días con Tmax > 35 °C, > 38 °C y > 40 °C durante enero–febrero.  
Este indicador es agronómicamente más informativo que la temperatura media porque cuantifica la exposición acumulada al estrés térmico.

In [ ]:
mascara_verano = df.index.month.isin([1, 2])
tmax_diaria_verano = (
    df.loc[mascara_verano, "temperature_2m"]
    .groupby(df.index[mascara_verano].date)
    .max()
)
tmax_diaria_verano.index = pd.to_datetime(tmax_diaria_verano.index)

dias35 = tmax_diaria_verano.groupby(tmax_diaria_verano.index.year).apply(lambda x: (x > 35).sum())
dias38 = tmax_diaria_verano.groupby(tmax_diaria_verano.index.year).apply(lambda x: (x > 38).sum())
dias40 = tmax_diaria_verano.groupby(tmax_diaria_verano.index.year).apply(lambda x: (x > 40).sum())

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(dias35.index, dias35.values, linewidth=2.2, marker="o", markersize=4, label=">35 °C")
ax.plot(dias38.index, dias38.values, linewidth=2.2, marker="o", markersize=4, label=">38 °C")
ax.plot(dias40.index, dias40.values, linewidth=2.2, marker="o", markersize=4, label=">40 °C")

ax.set_title(
    "Días con temperatura extrema durante el verano (enero–febrero)\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=14, fontweight="bold",
)
ax.set_ylabel("Número de días", fontsize=11)
ax.set_xlabel("Año", fontsize=11)
ax.grid(alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia.", ha="right",
            fontsize=9, color="gray")
plt.show()


## Distribución de Tmax por década (boxplots)

Permite evidenciar cambios en la distribución completa de las temperaturas máximas de verano, no solo en la media.  
Cada caja muestra la mediana, el rango intercuartílico y los valores atípicos de cada década.

In [ ]:
decadas_etiquetas = {
    1990: "1990–1999",
    2000: "2000–2009",
    2010: "2010–2019",
    2020: "2020–2024",
}
colores_decadas = {
    1990: "#AED6F1",
    2000: "#85C1E9",
    2010: "#E59866",
    2020: "#C0392B",
}

decada_serie = (tmax_diaria_verano.index.year // 10) * 10

datos_boxplot = [
    tmax_diaria_verano[decada_serie == d].values
    for d in sorted(decadas_etiquetas)
]
etiquetas     = [decadas_etiquetas[d] for d in sorted(decadas_etiquetas)]
colores_lista = [colores_decadas[d]   for d in sorted(decadas_etiquetas)]

fig, ax = plt.subplots(figsize=(10, 6))

bp = ax.boxplot(
    datos_boxplot,
    labels=etiquetas,
    patch_artist=True,
    widths=0.5,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.4),
    capprops=dict(linewidth=1.4),
    flierprops=dict(marker="o", markersize=4, linestyle="none", alpha=0.5),
)
for patch, color in zip(bp["boxes"], colores_lista):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.axhline(35, color="green", linewidth=1, linestyle="--", label="35 °C (límite óptimo inferior)")
ax.axhline(38, color="green", linewidth=1, linestyle="-.", label="38 °C (límite óptimo superior)")

ax.set_title(
    "Distribución de Tmax diaria en verano (ene–feb) por década\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=14, fontweight="bold",
)
ax.set_ylabel("Temperatura máxima diaria (°C)", fontsize=11)
ax.set_xlabel("Década", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4)
ax.legend(fontsize=10)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia.", ha="right",
            fontsize=9, color="gray")
plt.show()


## Anomalías térmicas respecto al período de referencia 1990–2020

Estándar climatológico de la OMM: la anomalía de cada año se expresa como la diferencia respecto al promedio del período base 1990–2020.  
Barras rojas = años más cálidos que el promedio; barras azules = años más frescos.

In [ ]:
ref_media   = calor_verano.loc[1990:2020].mean()
anomalias   = calor_verano - ref_media
colores_bar = ["#C0392B" if v >= 0 else "#2E86C1" for v in anomalias]

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(anomalias.index, anomalias.values, color=colores_bar, width=0.8)
ax.axhline(0, color="black", linewidth=0.9)

reg_an = linregress(anomalias.index.astype(float), anomalias.values)
ax.plot(
    anomalias.index,
    reg_an.intercept + reg_an.slope * anomalias.index.astype(float),
    color="black", linestyle="--", linewidth=1.5,
    label=f"Tendencia = {reg_an.slope:.3f} °C/año   (p={reg_an.pvalue:.3f})",
)

ax.set_title(
    "Anomalías de Tmax media en verano respecto al período base 1990–2020\n"
    "Jocolí, Lavalle, Mendoza",
    fontsize=14, fontweight="bold",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("Anomalía (°C)", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4)
ax.legend(fontsize=10)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia.", ha="right",
            fontsize=9, color="gray")
plt.show()

print(f"\nMedia de referencia 1990–2020 : {ref_media:.2f} °C")
print(f"Anomalía promedio 2021–2024   : {anomalias.loc[2021:].mean():+.2f} °C")


## Déficit hídrico anual (mm)

In [ ]:
años_dh = deficit_hidrico.index
tendencia_dh = np.polyfit(años_dh, deficit_hidrico.values, 1)
linea_tendencia_dh = np.poly1d(tendencia_dh)

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(años_dh, deficit_hidrico.values, color="#5b8db8", alpha=0.75,
       label="Deficit hídrico (mm)")

ax.axhline(600, color="#e07b39", linewidth=2, linestyle="--",
           label="600 mm (~6.000 m\u00b3/ha, mínimo plan de negocios)")

ax.plot(años_dh, linea_tendencia_dh(años_dh), color="#c0392b", linewidth=1.5,
        linestyle="-.", label=f"Tendencia ({tendencia_dh[0]:+.1f} mm/año)")

ax.set_title("Déficit hídrico anual (ET\u2080 - precipitación)\nJocolí, Lavalle, Mendoza (1990–2024)",
             fontsize=13)
ax.set_xlabel("Año")
ax.set_ylabel("mm / año")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Parámetros calibrados — exportar al Módulo 1

In [ ]:
from scipy import stats

UMBRAL_FRIO_CRITICO = 800

_dist, _params = "norm", stats.norm.fit(horas_frio.values)

PARAMS_CLIMA_JOCOLI = {
    "horas_frio_media":          float(horas_frio.mean()),
    "horas_frio_std":            float(horas_frio.std()),
    "horas_frio_dist":           _dist,
    "horas_frio_dist_params":    _params,
    "prob_anio_critico":         float((horas_frio < UMBRAL_FRIO_CRITICO).mean()),
    "prob_helada_tardia":        float((heladas > 0).mean()),
    "calor_verano_media":        float(calor_verano.mean()),
    "deficit_hidrico_media_mm":  float(deficit_hidrico.mean()),
}

print("PARAMS_CLIMA_JOCOLI = {")
for k, v in PARAMS_CLIMA_JOCOLI.items():
    print(f"    {k!r}: {v},")
print("}")